# Verification of residual and spectral bounds for the self-similar profile

This notebook computes rigorous bounds for several quantities associated with the self-similar profile $U$.

Specifically, we compute and print:

- The $L^2$ norm of the residual of the linear operator applied to $U$
- A rigorous upper bound for the $W^{k,∞} / L^{∞}$ norms of the matrix function $Q$ and its eigenvalues,
- Bounds for the symmetric gradient of $U$

These quantities are used to verify that the matrix function $Q$ cancels the negative part of sym($\nabla U$) (in the rigorous pointwise bound sense), which is a key condition in the stability verification.

Finally, we compute eigenvalue bounds on a refined mesh for further verification.

In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/julia code/residual_verification`


.Interval{Float64}
Number of threads = 56


## Coordinate systems and spectral representations

This notebook uses two different spectral representations for the velocity field $U$ and the matrix field $Q$.

### Velocity field $U$

The velocity field $U$ (stored in `UP`) is represented in the frequency domain with respect to the computational radial coordinate $\beta$ and angular coordinate $\theta$, where

$$
\beta = \arctan(r / a), \qquad \beta \in [0, \pi/2].
$$

The physical radius is recovered via

$$
r = a \tan(\beta).
$$

All truncation error bounds and interpolation operations for $U$ must therefore use the $\beta$-grid.


### Matrix field $Q$

The matrix field $Q$ is represented in the frequency domain with respect to the physical radius $r$ and $\theta$.

As a result, interpolation and truncation error bounds for $Q$ must be performed on the physical $r$-grid, not the $\beta$-grid.


### Summary

- $U$: spectral representation in $(\beta, \theta)$  
- $Q$: spectral representation in $(r, \theta)$  

This distinction is essential for correctly computing interpolation and truncation error bounds.

In [2]:
# Load the self-similar profile
UP = from_mat_file("../data/UP.mat", 2; domain="frequency")
M0, M1 = size(UP["u3"].freq_func)

# Load the Q matrix function 
Q = load_grad_U("../data/approx_Q.mat")
R_max = dt(32)
a = (Pi / dt(2)) * UP.scl_fac / R_max;

In [3]:
# Due to symmetry, only need to compute the residual on [0, Pi/2]x[0, Pi/2]
L0, L1, L_Q = Pi / dt(2), Pi / dt(2), one(dtype)

# Number of grid points in r and θ directions
N0, N1 = 8000, 4000
# Order of the Wk∞ norm needed for Q
k = 12

# Build the r and θ points as column‐vectors. We need to extend the grid
# by k points on each side for calculating the Wk∞ norm by finite difference.
r_pt_Q_ext = dt.(-k:N0+k) .*(L0/dt(N0))
θ_pt_ext = dt.(-k:N1+k) .*(L1/dt(N1))

# Interpolate Q on the extended grid and trim the border later
Q_ex_dict = interpolate(Q, r_pt_Q_ext, θ_pt_ext)
Q_dict = trim_border!(copy(Q_ex_dict), k);

### Spectral frequency bounds for $Q$

**Domain convention.**  
The coefficients of $Q$ are stored assuming the physical radial coordinate
$$
r \in [0,\ \pi/2].
$$
However, in the verification we use the normalized coordinate
$$
\hat r \in [0,\ 1], \qquad \hat r = \frac{2}{\pi}\, r \ \ \Longleftrightarrow\ \ r = \frac{\pi}{2}\hat r.
$$
Therefore, when rewriting $Q(r,\theta)$ as $Q\!\left(\frac{\pi}{2}\hat r,\theta\right)$, the radial oscillation in $\hat r$ is scaled by a factor $\pi/2$.

**Basis and effective wavenumbers.**  
The spectral basis for $Q$ uses odd/even harmonics, i.e.,
$$
\sin\big((2n-1)x\big), \qquad \cos\big(2n x\big),
$$
so the largest harmonic index is **twice** the nominal truncation index (up to the odd/even offset).  
Consequently, the effective maximum wavenumbers are taken as

- radial direction (in normalized $\hat r$):
  $$k_{\hat r,\max} \approx (2M_0)\cdot\frac{\pi}{2} = M_0\pi,$$

- angular direction:
  $$k_{\theta,\max} \approx 2M_1.$$

Accordingly, the frequency bounds used in the code are
$$
\texttt{Q\_freq\_lst} = [M_0\pi,\ 2M_1].
$$

In [4]:
Q_freq_lst = [dt(M0) * Pi, dt(M1 * 2)]

# A priori upper bound for the W^{k,∞} norm of Q. Need two more derivatives
# to compute the W^{k,∞} norm by finite difference.
Q_Wk∞ = norm_lst(Q_dict, "W$(k+2)Inf"; paras=Q_freq_lst, domain=[L_Q,L1])

# Compute the W^{k,∞} norm of Q on the extended grid for a sharper bound
paras_lst = Linf_to_paras(Q_Wk∞, k, Q_freq_lst)
Q_Wk∞ = norm_lst(Q_ex_dict, "W$(k)Inf"; paras=paras_lst, domain=[L_Q,L1])

# W^{2,∞} norm bounds for each component of Q
Q_dββ_L∞, Q_dθθ_L∞ = norm(Q_Wk∞[3, 1]), norm(Q_Wk∞[3, 2])

# Save the W^{k,∞} norm bounds of Q for the error estimate of numerical integrals
Q_Wk∞_mat = hcat(Q_Wk∞[:, 1]...)'
save_matrix("../data/Q_bound.mat", Dict("Q_WkInf" => Q_Wk∞_mat))

In [5]:
# Build the r and θ points as column‐vectors
N0, N1 = 12000, 6000
β_pt = dt.(0:N0) .*(L0/dt.(N0))
θ_pt = dt.(0:N1) .*(L1/dt.(N1))

# Generate the operators mat and compute the residual
lin_mat = get_operator_matrix(UP, β_pt, θ_pt, "linear")
err = mul_A_gradB(UP, UP, β_pt, θ_pt) - apply(lin_mat, UP)

# Compute Q on the mesh
Q_dict = interpolate(Q, β_pt, θ_pt)

# Compute the symmetric part of the gradient
grad_sym_mat = get_operator_matrix(UP, β_pt, θ_pt, "gradient_sym")
grad_sym_UP = apply(grad_sym_mat, UP; new_keys=["A11", "A12", "A13", "A22", "A23", "A33"]);

In [6]:
# Compute the W^{k,∞} norm of the residual.
# The wavenumber of UP is 2*M0. The residual has terms like U \cdot \nabla U,
# so the wavenumber is 4*M0.
freq_lst = [dt(4 * M0 + 10), dt(4 * M1 + 10)]
W_norms = norm_lst(err, "W$(k)Inf"; paras=freq_lst, domain=[L0,L1])

# Calculate the L2 norm of the residual with rigorous error bounds
L2_norms = norm_lst(err, "L2"; paras=W_norms)
println("L^2 error of linear operator: ", L2_norms)

L^2 error of linear operator: [6.40737e-7, 6.40861e-7]_trv


In [7]:
# Compute the maximum eigenvalue of Q with a rigorous bound
eig_val = eigen_calc(Q_dict)
# L∞_fac accounts for the truncation/aliasing error when converting W^{k,∞} bounds
# to pointwise eigenvalue bounds on the discrete grid (see Section 7 of the paper).
L∞_fac = dt(1) - (((L0/dt(N0)) * freq_lst[1])^2 + ((L1/dt(N1)) * freq_lst[2])^2) / dt(8)
println("Max eigenvalue: ", maximum(eig_val) / L∞_fac)

Max eigenvalue: [4.49794, 4.49794]_trv


In [8]:
# Estimate the W^{2,∞} norm of the symmetric gradient
grad_norms = norm_lst(grad_sym_UP, "LInf"; paras=freq_lst, domain=[L0,L1])
grad_dββ_L∞, grad_dθθ_L∞ = norm(grad_norms) * freq_lst[1]^2, norm(grad_norms) * freq_lst[2]^2;

In [9]:
# Build the r and θ points as column‐vectors for eigenvalue calculation
N0, N1 = 24000, 12000
Beta = atan(R_max / UP.scl_fac)
β_pt_lst = [dt.(0:N0) .* (Beta / dt(N0)),
    Beta .+ dt.(0:N0) .* ((L0 - Beta) / dt(N0))]
θ_pt = dt.(0:N1) .*(L1/dt(N1));

In [10]:
# Truncation error bound for eigenvalues:
# (A) discretization error from sym(∇U) on (β, θ)-grid
# (B) additional discretization error from adding Q on (r, θ)-grid (part 1 only)
for i in 1:2
    β_pt = β_pt_lst[i]

    # Compute the symmetric part of the gradient
    grad_sym_mat = get_operator_matrix(UP, β_pt, θ_pt, "gradient_sym")
    grad_res = apply(grad_sym_mat, UP; new_keys=["A11", "A12", "A13", "A22", "A23", "A33"])
    
    # Upper bound of the truncation error in eigenvalue calculation
    h_β, h_θ = β_pt[2] - β_pt[1], θ_pt[2] - θ_pt[1]
    delta_eig_val = (h_β^2 * grad_dββ_L∞ + h_θ^2 * grad_dθθ_L∞) / dt(8)
    
    if i == 1
        r_pt = tan.(β_pt) .* a;  # Convert to physical space
        # For the first part, add Q on [0, Beta]x[0, π/2]
        add_equal!(grad_res, interpolate(Q, r_pt, θ_pt))

        # Estimate the additional truncation error from Q
        # Note: r-grid is non-uniform after r = a*tan(β), so use the maximum step size.
        h_r = maximum(diff(r_pt))
        delta_eig_val += (h_r^2 * Q_dββ_L∞ + h_θ^2 * Q_dθθ_L∞) / dt(8)
    end

    # Compute the minimum eigenvalue with rigorous bounds
    eig_val = eigen_calc(grad_res)
    min_eig = minimum(eig_val) - delta_eig_val
    println("Lower bound of the minimum eigenvalue part ", i, " : ", min_eig)
end

Lower bound of the minimum eigenvalue part 1 : [-0.198578, -0.198577]_trv
Lower bound of the minimum eigenvalue part 2 : [-0.17948, -0.17948]_trv
